<a href="https://colab.research.google.com/github/aatishsharmacs24-star/AI-lab-1-/blob/main/Week-8/1WA24CSOO5_Week8_FOL_ForwardReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
facts = {
    ("Owns", "A", "T1"),
    ("Missile", "T1"),
    ("American", "Robert"),
    ("Enemy", "A", "America")
}

rules = [
    ([("Missile", "x")], ("Weapon", "x")),
    ([("Missile", "x"), ("Owns", "A", "x")],
     ("Sells", "Robert", "x", "A")),
    ([("Enemy", "x", "America")], ("Hostile", "x")),
    ([("American", "p"), ("Weapon", "q"),
      ("Sells", "p", "q", "r"), ("Hostile", "r")],
     ("Criminal", "p"))
]

variables = {"x", "p", "q", "r"}

def apply_substitution(term, subst):
    if isinstance(term, str):
        return subst.get(term, term)

    if isinstance(term, tuple):
        return tuple(apply_substitution(t, subst) for t in term)

    return term

def unify(x, y, subst):
    x = apply_substitution(x, subst)
    y = apply_substitution(y, subst)

    if x == y:
        return subst

    if isinstance(x, str) and x in variables:
        subst[x] = y
        return subst

    if isinstance(y, str) and y in variables:
        subst[y] = x
        return subst

    if isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y):
            return None

        for a, b in zip(x, y):
            subst = unify(a, b, subst)
            if subst is None:
                return None

        return subst

    return None

def match_premises(premises, facts, subst=None):
    if subst is None:
        subst = {}

    if not premises:
        return [subst]

    first = apply_substitution(premises[0], subst)
    remaining = premises[1:]
    results = []

    for fact in facts:
        new_subst = unify(first, fact, subst.copy())

        if new_subst is not None:
            results.extend(
                match_premises(remaining, facts, new_subst)
            )

    return results

def forward_chaining(facts, rules):
    facts = set(facts)
    changed = True

    while changed:
        changed = False

        for premises, conclusion in rules:
            substitutions = match_premises(premises, facts)

            for subst in substitutions:
                new_fact = apply_substitution(conclusion, subst)

                if new_fact not in facts:
                    facts.add(new_fact)
                    print("Inferred:", new_fact)
                    changed = True

    return facts

result = forward_chaining(facts, rules)

query = ("Criminal", "Robert")

print("\nQuery:", query)
print("Proved:", query in result)

Inferred: ('Weapon', 'T1')
Inferred: ('Sells', 'Robert', 'T1', 'A')
Inferred: ('Hostile', 'A')
Inferred: ('Criminal', 'Robert')

Query: ('Criminal', 'Robert')
Proved: True
